This cell installs the necessary Python libraries for the project, including `wrds`, `psycopg2-binary`, `pandas`, `numpy`, `pyarrow`, and `pandas_market_calendars`.

In [ ]:
# Dependencies are installed in the local environment; see requirements.txt.
# (Colab original ran: %pip install -q wrds psycopg2-binary pandas numpy pyarrow pandas_market_calendars)

This cell imports essential libraries like pandas and numpy. It then clones a GitHub repository containing project data and moves a CSV file from the repository to the working directory. It also includes commented-out code for uploading local evaluation scripts, which are not currently being used.

In [ ]:
# --- local adaptation of the Colab notebook (upstream: Corrected_SIADS_699_Capstone_Features_v4) ---
# Anchored to the repo root so ./data resolves to the project's own data directory.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(_root); sys.path.insert(0, str(_root))

# FRED key from .env (gitignored). pandas_datareader's FRED reader does not require one,
# but it is wired up so the notebook does not depend on that staying true.
for _line in (Path(".env").read_text(encoding="utf-8").splitlines() if Path(".env").exists() else []):
    if "=" in _line and not _line.startswith("#"):
        _k, _v = _line.split("=", 1); os.environ.setdefault(_k.strip(), _v.strip())

# THE EXPERIMENT KNOB. Upstream used [2022, 2023, 2024, 2025] -- four folds, ~1,000 test
# observations. The panel spans 2015-2025, so the training data was never the constraint; the
# out-of-sample evaluation was. Starting at 2019 avoids XLC's June-2018 listing, which would
# otherwise shift the cross-sectional benchmark (excess = return - cross-sectional mean) partway
# through a fold.
TEST_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
print(f"repo root: {_root}")
print(f"TEST_YEARS = {TEST_YEARS}  ({len(TEST_YEARS)} folds)")

import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import copy
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import yfinance as yf
from __future__ import annotations
import random
from dataclasses import dataclass, field
from typing import Callable, Iterator, Sequence
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
import wrds
from datetime import time
from pathlib import Path
import pandas_datareader.data as web
import warnings
from scipy.stats import ConstantInputWarning


print("Environment setup complete.")

In [ ]:
file_path = "data/model_daily_panel.csv"
panel_df = pd.read_csv(file_path, parse_dates=["session_date"])
display(panel_df.head())

In [16]:
DATE, TICKER = "session_date", "ticker"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    except ImportError: pass

@dataclass(frozen=True)
class Fold:
    test_year: int
    train_dates: np.ndarray
    val_dates: np.ndarray
    test_dates: np.ndarray

    def __str__(self) -> str:
        rng = lambda d: f"{pd.Timestamp(d[0]).date()}..{pd.Timestamp(d[-1]).date()}" if len(d) else "empty"
        return f"Fold {self.test_year}: train {rng(self.train_dates)} | val {rng(self.val_dates)} | test {rng(self.test_dates)}"

class PurgedWalkForward:
    def __init__(self, test_years=(2022, 2023, 2024, 2025), label_horizon=1, embargo=None, val_sessions=126):
        self.test_years, self.label_horizon, self.val_sessions = list(test_years), int(label_horizon), int(val_sessions)
        self.embargo = int(embargo) if embargo is not None else int(label_horizon)

    def split(self, panel: pd.DataFrame) -> Iterator[Fold]:
        all_dates = np.sort(panel[DATE].unique())
        for yr in self.test_years:
            start, end = pd.Timestamp(f"{yr}-01-01"), pd.Timestamp(f"{yr}-12-31")
            test_dates = all_dates[(all_dates >= start) & (all_dates <= end)]
            if not len(test_dates): continue
            history = all_dates[all_dates < start]
            if self.embargo > 0: history = history[:-self.embargo] if len(history) > self.embargo else history[:0]
            n_val = min(self.val_sessions, max(len(history) // 5, 1))
            val_dates, train_pool = history[-n_val:], history[:-n_val]
            train_dates = train_pool[:-self.label_horizon] if self.label_horizon > 1 and len(train_pool) > self.label_horizon else train_pool
            if len(train_dates): yield Fold(yr, train_dates, val_dates, test_dates)

def fit_scaler(train_df, feature_cols, skip=("sector_",)):
    cols = [c for c in feature_cols if not str(c).startswith(skip)]
    return StandardScaler().fit(train_df[cols]), cols

def build_sequences(df, feature_cols, target_col, seq_len=21):
    seqs, labels, keys = [], [], []
    df = df.sort_values([TICKER, DATE])
    for tkr, g in df.groupby(TICKER, sort=False):
        X, y, d = g[list(feature_cols)].values, g[target_col].values, g[DATE].values
        if len(g) < seq_len: continue
        for end in range(seq_len - 1, len(g)):
            if np.isnan(y[end]) or np.isnan(X[end-seq_len+1:end+1]).any(): continue
            seqs.append(X[end-seq_len+1:end+1])
            labels.append(y[end]); keys.append((tkr, d[end]))
    return np.array(seqs, dtype=np.float32), np.array(labels, dtype=np.float32), pd.DataFrame(keys, columns=[TICKER, DATE])

def date_block_bootstrap_auc(y, p, dates, n_boot=500, block_days=21, seed=10):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"y": y, "p": p, "d": pd.Series(dates).astype(str).values})
    uniq = np.sort(df["d"].unique())
    n_blocks, by_date = max(len(uniq) // block_days, 1), {d: g for d, g in df.groupby("d")}
    stats_ = []
    for _ in range(n_boot):
        starts = rng.integers(0, max(len(uniq) - block_days, 1), size=n_blocks)
        picked = np.concatenate([uniq[s:s+block_days] for s in starts])
        sample = pd.concat([by_date[d] for d in picked if d in by_date])
        if sample["y"].nunique() >= 2: stats_.append(roc_auc_score(sample["y"], sample["p"]))
    return (roc_auc_score(y, p), np.percentile(stats_, [2.5, 97.5])) if stats_ else (roc_auc_score(y, p), (0, 0))

def long_short_backtest(pred, fwd_return, dates, n_side=3, cost_bps=0.0002):
    df = pd.DataFrame({"p": pred, "r": fwd_return, "d": dates}).dropna()
    rets, to, prev_w = [], [], None
    for d, g in df.groupby("d"):
        if len(g) < 2 * n_side: continue
        g = g.sort_values("p")
        w = pd.Series(0.0, index=g.index)
        w.iloc[-n_side:], w.iloc[:n_side] = 1.0/n_side, -1.0/n_side
        rets.append((w * g["r"]).sum())
        to.append((w - prev_w.reindex(w.index).fillna(0)).abs().sum() if prev_w is not None else w.abs().sum())
        prev_w = w
    r, to = np.array(rets), np.array(to)
    if len(r) < 2: return {}
    net, ann = r - to * cost_bps, np.sqrt(252)
    return {"gross_sharpe": r.mean()/r.std(ddof=1)*ann, "net_sharpe": net.mean()/net.std(ddof=1)*ann,
            "gross_ann_return": r.mean()*252, "net_ann_return": net.mean()*252}


In [17]:
def information_coefficient(preds, returns, dates):
    df = pd.DataFrame({'p': preds, 'r': returns, 'd': dates}).dropna()
    if not len(df): return {}
    ic = df.groupby('d').apply(lambda g: g['p'].corr(g['r'], method='spearman') if len(g) > 2 else np.nan).dropna()
    if len(ic) < 2: return {"mean_ic": np.nan, "t_stat": np.nan}
    return {"mean_ic": ic.mean(), "t_stat": stats.ttest_1samp(ic, 0)[0]}

def example_logistic_fit_predict(train_df, val_df, test_df, feature_cols, target_col, seed=10):
    set_seed(seed)
    scaler = StandardScaler().fit(train_df[feature_cols])
    Xtr, Xte = scaler.transform(train_df[feature_cols]), scaler.transform(test_df[feature_cols])
    clf = LogisticRegression(C=0.1, max_iter=2000, random_state=seed).fit(Xtr, train_df[target_col].values)
    return clf.predict_proba(Xte)[:, 1]

def run_walk_forward(panel, feature_cols, target_col, fit_predict, splitter, config_name, seeds=(10,), fwd_return_col=None):
    all_rows = []
    for fold in splitter.split(panel):
        tr = panel[panel[DATE].isin(fold.train_dates)].copy()
        va = panel[panel[DATE].isin(fold.val_dates)].copy()
        te = panel[panel[DATE].isin(fold.test_dates)].copy()

        needed = list(feature_cols) + [target_col]
        if fwd_return_col: needed.append(fwd_return_col)
        tr, va, te = (d.dropna(subset=needed) for d in (tr, va, te))
        if min(len(tr), len(va), len(te)) == 0: continue

        for seed in seeds:
            p = fit_predict(tr, va, te, feature_cols, target_col, seed)
            auc, _ = date_block_bootstrap_auc(te[target_col].values, p, te[DATE].values, seed=seed)

            row = {"config": config_name, "fold": fold.test_year, "seed": seed, "auc": auc}
            if fwd_return_col:
                ic = information_coefficient(p, te[fwd_return_col], te[DATE])
                ls = long_short_backtest(p, te[fwd_return_col], te[DATE])
                row.update({f"ic_{k}": v for k, v in ic.items()})
                row.update({f"ls_{k}": v for k, v in ls.items()})
            all_rows.append(row)
    return pd.DataFrame(all_rows)


## **Point-in-Time and Econometric Taxonomy**

This cell focuses on the final deep learning evaluation using the Sector Autoformer for models M0 through M4. It sets up the custom Autoformer architecture, including series decomposition and auto-correlation mechanisms. It also defines functions for sequence windowing and training logic, then executes a purged walk-forward evaluation for each model configuration, presenting a final daily ablation matrix.

This section explains the point-in-time and econometric taxonomy used in the analysis.

In [18]:
print("PIPELINE")

DATA_DIR, CACHE_DIR = Path("./data"), Path("./data/cache")
for d in (DATA_DIR, CACHE_DIR): d.mkdir(parents=True, exist_ok=True)

START_DATE, END_DATE = pd.Timestamp("2015-01-01"), pd.Timestamp("2025-12-31")
SECTORS = ["XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY"]
MARKET_TZ = "America/New_York"

def load_sector_prices():
    cache = CACHE_DIR / "sector_prices.parquet"
    if cache.exists(): return pd.read_parquet(cache)

    raw = yf.download(SECTORS, start=START_DATE, end=END_DATE + pd.Timedelta(days=1), auto_adjust=True, progress=False, group_by="ticker")
    frames = []
    for t in SECTORS:
        sub = raw[t].reset_index().rename(columns=lambda c: str(c).lower()).rename(columns={"date": "session_date"})
        sub["ticker"] = t
        frames.append(sub[["session_date", "ticker", "open", "high", "low", "close", "volume"]])

    df = pd.concat(frames).dropna(subset=["open", "close"]).sort_values(["ticker", "session_date"])
    df["session_date"] = pd.to_datetime(df["session_date"]).dt.tz_localize(None).dt.normalize()
    df.reset_index(drop=True).to_parquet(cache, index=False)
    return df

def decompose_returns(df):
    df = df.sort_values(["ticker", "session_date"]).copy()
    g = df.groupby("ticker")

    df["prev_close"] = g["close"].shift(1)
    df["overnight_return"] = (df["open"] / df["prev_close"]) - 1.0
    df["intraday_return"] = (df["close"] / df["open"]) - 1.0
    df["c2c_return"] = (df["close"] / df["prev_close"]) - 1.0
    df["volume_log"] = np.log1p(df["volume"].clip(lower=0))
    df["dollar_volume"] = df["close"] * df["volume"]

    for col in ["overnight_return", "intraday_return", "c2c_return"]:
        mkt = df.groupby("session_date")[col].transform("mean")
        df[f"mkt_{col}"] = mkt
        df[f"excess_{col}"] = df[col] - mkt

    df["target_overnight"] = (df["excess_overnight_return"] > 0).astype(float)
    return df.dropna(subset=["target_overnight"]).reset_index(drop=True)

def assign_news_windows(news_df, session_dates):
    df = news_df.copy()
    ts = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
    df = df.loc[ts.notna()].copy()
    ny_time = ts.loc[df.index].dt.tz_convert(MARKET_TZ)

    df["et_date"], tod = ny_time.dt.normalize().dt.tz_localize(None), ny_time.dt.time
    trading_days = pd.DatetimeIndex(np.sort(session_dates))

    is_pre_open = tod < time(9, 15)
    is_session = (tod >= time(9, 15)) & (tod < time(15, 45))
    is_blackout = tod >= time(15, 45)

    idx_curr = trading_days.searchsorted(df["et_date"].values, side="left")
    idx_next = trading_days.searchsorted(df["et_date"].values, side="right")

    n_days = len(trading_days)
    sess_curr = np.where(idx_curr < n_days, trading_days.values[np.clip(idx_curr, 0, n_days - 1)], np.datetime64("NaT"))
    sess_next = np.where(idx_next < n_days, trading_days.values[np.clip(idx_next, 0, n_days - 1)], np.datetime64("NaT"))

    is_trading_day = df["et_date"].isin(trading_days).values
    df["session_date"] = pd.to_datetime(np.where(is_pre_open.values | ~is_trading_day, sess_curr, np.where(is_blackout.values, sess_next, sess_curr)))
    df["window"] = np.where(is_session.values & is_trading_day, "session", "overnight")

    return df.loc[df["session_date"].notna()].copy()

def build_theme_panel(aligned_news, themes, window="overnight"):
    df = aligned_news.loc[aligned_news["window"] == window].copy()
    df["theme"] = df["group"].fillna(df["topic"]).fillna("unclassified").str.lower().str.strip()
    df = df.loc[df["theme"].isin(themes)]

    ess = pd.to_numeric(df["event_sentiment_score"], errors="coerce")
    df = df.assign(ess=ess, novelty=pd.to_numeric(df.get("event_similarity_days", 0), errors="coerce"))

    agg = df.groupby(["session_date", "theme"]).agg(ess=("ess", "mean"), disp=("ess", "std"), vol=("ess", "size"), novelty=("novelty", "mean")).reset_index()
    agg["vol"] = np.log1p(agg["vol"])
    agg["disp"] = agg["disp"].fillna(0.0)

    wide = agg.pivot(index="session_date", columns="theme")
    wide.columns = [f"{theme}__{stat}" for stat, theme in wide.columns]

    for c in wide.columns:
        wide[c] = wide[c].fillna(0.0) if c.split("__")[1] in {"vol", "disp"} else wide[c].ffill().fillna(0.0)
    return wide.sort_index()

def engineer_market_features(df):
    df = df.sort_values(["ticker", "session_date"]).copy()
    g = df.groupby("ticker")

    for lag in [1, 5, 20]: df[f"return_lag_{lag}"] = g["c2c_return"].shift(lag)
    df["return_mean_5"] = g["c2c_return"].transform(lambda s: s.shift(1).rolling(5, min_periods=5).mean())
    df["return_mean_20"] = g["c2c_return"].transform(lambda s: s.shift(1).rolling(20, min_periods=20).mean())
    df["return_std_20"] = g["c2c_return"].transform(lambda s: s.shift(1).rolling(20, min_periods=20).std(ddof=0))
    df["volume_log_lag_1"] = g["volume_log"].shift(1)
    df["overnight_lag_1"] = g["overnight_return"].shift(1)
    df["intraday_lag_1"] = g["intraday_return"].shift(1)

    amihud = np.log1p((df["c2c_return"].abs() / (df["close"] * df["volume"] + 1e-8)) * 1e9)
    df["amihud_illiq_log_lag_1"] = df.assign(a=amihud).groupby("ticker")["a"].shift(1)

    base_features = [c for c in df.columns if "lag" in c or "mean" in c or "std" in c]
    for col in base_features:
        df[f"cs_{col}"] = df.groupby("session_date")[col].transform(lambda x: (x - x.mean()) / (x.std(ddof=1) + 1e-8))

    return df, [f"cs_{col}" for col in base_features]

def rolling_sector_betas(rets_wide, theme_wide, window=252, min_periods=126):
    idx = rets_wide.index.intersection(theme_wide.index)
    R, T = rets_wide.loc[idx], theme_wide.loc[idx]
    var_t, mean_t = T.rolling(window, min_periods=min_periods).var(ddof=0), T.rolling(window, min_periods=min_periods).mean()

    betas = {}
    for tkr in R.columns:
        cov = T.mul(R[tkr], axis=0).rolling(window, min_periods=min_periods).mean() - mean_t.mul(R[tkr].rolling(window, min_periods=min_periods).mean(), axis=0)
        betas[tkr] = (cov / var_t.replace(0.0, np.nan)).shift(1)
    return betas

def project_narrative(theme_panel, betas, prefix="narr"):
    themes = sorted({c.split("__")[0] for c in theme_panel.columns})
    ess = theme_panel[[f"{t}__ess" for t in themes]].rename(columns=lambda c: c.split("__")[0])
    vol = theme_panel[[f"{t}__vol" for t in themes]].rename(columns=lambda c: c.split("__")[0])
    disp = theme_panel[[f"{t}__disp" for t in themes]].rename(columns=lambda c: c.split("__")[0])
    nov = theme_panel[[f"{t}__novelty" for t in themes]].rename(columns=lambda c: c.split("__")[0])

    vol_mu = vol.rolling(60, min_periods=30).mean().shift(1)
    vol_sd = vol.rolling(60, min_periods=30).std(ddof=0).shift(1)
    attention_z = (vol - vol_mu) / (vol_sd + 1e-8)

    rows = []
    for tkr, b in betas.items():
        b = b.reindex(theme_panel.index)[themes]
        w = b.abs().sum(axis=1).replace(0.0, np.nan)
        out = pd.DataFrame(index=theme_panel.index)
        out[f"{prefix}_score"] = (b * ess).sum(axis=1) / w
        out[f"{prefix}_attention"] = (b.abs() * attention_z).sum(axis=1) / w
        out[f"{prefix}_dispersion"] = (b.abs() * disp).sum(axis=1) / w
        out[f"{prefix}_novelty"] = (b.abs() * nov).sum(axis=1) / w
        out["ticker"] = tkr
        rows.append(out.reset_index().rename(columns={"index": "session_date"}))

    long = pd.concat(rows, ignore_index=True).sort_values(["ticker", "session_date"])
    g = long.groupby("ticker")[f"{prefix}_score"]
    mu, sd = g.transform(lambda s: s.shift(1).rolling(60, min_periods=30).mean()), g.transform(lambda s: s.shift(1).rolling(60, min_periods=30).std(ddof=0))
    long[f"{prefix}_surprise"] = (long[f"{prefix}_score"] - mu) / (sd + 1e-8)
    long[f"{prefix}_score_x_attention"] = long[f"{prefix}_surprise"] * long[f"{prefix}_attention"]
    return long.reset_index(drop=True)

# WRDS RavenPack
NEWS_COLUMNS = ["timestamp_utc", "rp_story_id", "rp_entity_id", "entity_name", "relevance",
                "event_sentiment_score", "event_relevance", "event_similarity_days",
                "topic", "group", "type", "category"]

def get_institutional_sources(db):
    """Rank-1, non-blog news sources."""
    raw = db.raw_sql("SELECT rp_entity_id, data_type, data_value FROM rpna.rpa_source_list "
                     "WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')")
    wide = raw.pivot(index="rp_entity_id", columns="data_type", values="data_value")
    wide.columns.name = None
    wide = wide.rename(columns={"ENTITY_NAME": "source_name", "PUBLICATION_TYPE": "source_type", "SOURCE_RANK": "source_rank"}).reset_index()
    wide["source_rank"] = pd.to_numeric(wide["source_rank"], errors="coerce")
    return wide[wide["source_rank"].isin([1]) & ~wide["source_type"].isin(["BLOG"])].reset_index(drop=True)

def extract_macro_news(db, source_ids, years=range(2015, 2026)):
    id_list = ", ".join(f"'{s}'" for s in source_ids)
    cols = ", ".join(f'"{c}"' if c in {"group", "type"} else c for c in NEWS_COLUMNS)
    frames = []
    for year in years:
        cache = CACHE_DIR / f"macro_news_{year}.parquet"
        if cache.exists():
            frames.append(pd.read_parquet(cache)); continue
        query = f"""SELECT {cols} FROM rpna.rpa_djpr_global_macro_{year}
                    WHERE relevance >= 90 AND event_relevance >= 90
                      AND event_sentiment_score IS NOT NULL AND rp_source_id IN ({id_list})"""
        df = db.raw_sql(query)
        df.to_parquet(cache, index=False); frames.append(df)
    return pd.concat(frames, ignore_index=True)

SECTOR_KEYWORDS = {
    "XLK": ["semiconductor", "software", "cloud computing", "hardware", "chip"],
    "XLV": ["pharmaceutical", "biotech", "hospital", "fda", "drug approval", "medicare"],
    "XLF": ["bank", "insurer", "credit", "lending", "capital markets", "basel"],
    "XLC": ["telecom", "streaming", "advertising", "social media", "media"],
    "XLY": ["retail", "consumer discretionary", "auto", "e-commerce", "restaurant"],
    "XLI": ["aerospace", "defense", "machinery", "logistics", "freight", "industrial"],
    "XLP": ["consumer staples", "grocery", "beverage", "tobacco", "household products"],
    "XLE": ["oil", "gas", "opec", "refinery", "crude", "drilling"],
    "XLU": ["utility", "electricity", "power grid", "renewable generation"],
    "XLB": ["mining", "chemicals", "steel", "copper", "materials"],
    "XLRE": ["reit", "real estate", "commercial property", "mortgage rate"],
}

def attribute_sentiment_to_sectors(aligned_news, keywords=SECTOR_KEYWORDS, window="overnight"):
    df = aligned_news.loc[aligned_news["window"] == window].copy()
    text = ""
    for c in ["entity_name", "topic", "group", "category"]:
        if c in df.columns: text = text + df[c].fillna("").astype(str) + " "
    text = text.str.lower()
    df["ess"] = pd.to_numeric(df["event_sentiment_score"], errors="coerce")

    frames = []
    for ticker, terms in keywords.items():
        hit = df.loc[text.str.contains("|".join(terms), regex=True, na=False)].copy()
        if hit.empty: continue
        hit["ticker"] = ticker
        frames.append(hit[["session_date", "ticker", "ess"]])
    if not frames:
        raise ValueError("no sector attributions produced; check keywords / text fields")

    attributed = pd.concat(frames, ignore_index=True)
    agg = attributed.groupby(["session_date", "ticker"]).agg(kw_sent=("ess", "mean"), kw_vol=("ess", "size")).reset_index()
    agg["kw_vol"] = np.log1p(agg["kw_vol"])
    return agg

def add_kw_surprise(panel, attributed, window=60):
    p = panel.merge(attributed, on=["session_date", "ticker"], how="left")
    p[["kw_sent", "kw_vol"]] = p[["kw_sent", "kw_vol"]].fillna(0.0)   # no attributed news -> neutral
    p = p.sort_values(["ticker", "session_date"])

    def surprise(col):
        g = p.groupby("ticker")[col]
        mu = g.transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
        sd = g.transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).std(ddof=0))
        return (p[col] - mu) / (sd + 1e-8)

    p["kw_sent_surprise"] = surprise("kw_sent")
    p["kw_attention"] = surprise("kw_vol")
    p["kw_sent_x_attention"] = p["kw_sent_surprise"] * p["kw_attention"]
    return p.reset_index(drop=True)

KW_FEATURES = ["kw_sent_surprise", "kw_sent_x_attention"]

print("Market Data")
market = decompose_returns(load_sector_prices())

print("RavenPack Data")
NEWS_CACHE = CACHE_DIR / "macro_news_all.parquet"
if NEWS_CACHE.exists():
    raw_news = pd.read_parquet(NEWS_CACHE)
    print(f"   Loaded cached macro news: {len(raw_news):,} rows")
else:
    db = wrds.Connection()
    sources = get_institutional_sources(db)
    print(f"   Institutional sources identified: {len(sources)}")
    raw_news = extract_macro_news(db, sources["rp_entity_id"].tolist())
    raw_news.to_parquet(NEWS_CACHE, index=False)
    db.close()

print("Aligning News & Taxonomy")
aligned_news = assign_news_windows(raw_news, np.sort(market["session_date"].unique()))
themes = aligned_news.loc[aligned_news["session_date"] <= pd.Timestamp("2021-12-31"), "group"].fillna(aligned_news["topic"]).fillna("unclassified").str.lower().str.strip().value_counts().head(24).index.tolist()
theme_overnight = build_theme_panel(aligned_news, themes, "overnight")

print("Features & Narratives")
market_panel, MARKET_FEATURES = engineer_market_features(market)

excess_wide = market_panel.pivot(index="session_date", columns="ticker", values="excess_overnight_return")
theme_wide = theme_overnight[[c for c in theme_overnight.columns if c.endswith("__ess")]].rename(columns=lambda c: c.split("__")[0])
on_betas = rolling_sector_betas(excess_wide, theme_wide)
narr_on_panel = project_narrative(theme_overnight, on_betas, prefix="narr_on")
NARR_ON_FEATURES = [c for c in narr_on_panel.columns if c.startswith("narr_on_")]

print("Panel")
panel = market_panel.merge(narr_on_panel, on=["session_date", "ticker"], how="left").dropna(subset=MARKET_FEATURES + NARR_ON_FEATURES).reset_index(drop=True)

print("Sector sentiment by keyword")
attributed = attribute_sentiment_to_sectors(aligned_news, window="overnight")
panel = add_kw_surprise(panel, attributed)

M0_FEATURES = MARKET_FEATURES
M1_FEATURES = MARKET_FEATURES + NARR_ON_FEATURES
M2_FEATURES = M0_FEATURES + ["proj_sentiment_score"]
M3_FEATURES = M1_FEATURES + ["proj_sentiment_score"]
M4_FEATURES = M3_FEATURES + ["proj_vix_score", "proj_treasury_10y_score", "proj_fed_funds_score", "proj_crude_oil_score", "proj_copper_score", "proj_spread_2y10y_score"]
M5_FEATURES = M4_FEATURES + KW_FEATURES

print(f"M0-M5 Panels done. Rows: {len(panel):,}")

PIPELINE
Market Data
RavenPack Data
   Loaded cached macro news: 1,199,890 rows
Aligning News & Taxonomy
Features & Narratives
Panel
Sector sentiment by keyword
M0-M5 Panels done. Rows: 27,635


2015-2025

This cell runs the end-to-end pipeline for data extraction, alignment, and taxonomy decomposition. It defines configurations and paths, then proceeds with market data extraction and return decomposition, WRDS RavenPack extraction, and point-in-time alignment and theme construction. It concludes by assembling the final panel for further analysis.

This cell performs the final panel assembly by projecting exposure and interactions. It defines functions for rolling sector betas and narrative projection, then builds overnight and session features. Finally, it engineers base market features, merges all projected and market features into a single panel, and asserts cross-sectional variance for diagnostic purposes.

This cell assembles the M2-M4 models by incorporating macroeconomic and FinBERT exposure projections. It fetches 10-year macro data from FRED and Yahoo Finance, loads FinBERT data, and then applies a projection machinery to create sector-specific projected features from market-wide macroeconomic and sentiment data. The final panel is then cleaned, and feature sets for M0-M4 are defined.

**Model Features Breakdown**


*   **M0 Market Baseline (10 features):** This model uses strictly engineered, cross-sectionally z-scored market features to prevent target leakage. The features include:
Lagged close-to-close returns (1, 5, and 20 days).
Mean returns (5 and 20 days).
Return standard deviation (20 days).
Lagged log volume.
Lagged overnight return.
Lagged intraday return.
Lagged log Amihud illiquidity.
*   **M1 Narrative Projected (16 features):** Includes all 10 M0 Market Baseline features, plus 6 overnight narrative features projected onto the sectors:
Narrative score.
Narrative attention.
Narrative dispersion.
Narrative novelty.
Narrative surprise (calculated over a 60-day rolling window).
Narrative score and attention interaction term (score_x_attention).
*   **M2 FinBERT Projected (11 features):** Includes all 10 M0 Market Baseline features, plus 1 projected FinBERT sentiment score feature.
*   **M3 Multimodal Projected (17 features):** Combines the 16 M1 Narrative features with the 1 projected FinBERT sentiment score.
*   **M4 Macro Projected (23 features):** Includes all 17 M3 Multimodal features, plus 6 projected macroeconomic variables:
VIX score.
10-year Treasury score.
Fed Funds score.
Crude oil score.
Copper score.
2-year/10-year Treasury spread score.
*   **M5 Sector Attribution (25 features):** Combines all 23 M4 features plus 2 keyword attribution features (sentiment surprise and sentiment x attention).














In [19]:
print("M2-M5 MACRO & FINBERT EXPOSURE")

fred_tickers = {
    'VIXCLS': 'ts_vix',
    'DGS10': 'ts_treasury_10y',
    'DFF': 'ts_fed_funds',
    'T10Y2Y': 'ts_spread_2y10y'
}
macro_fred = web.DataReader(list(fred_tickers.keys()), 'fred', '2015-01-01', '2025-12-31')
macro_fred = macro_fred.rename(columns=fred_tickers)

macro_yf = yf.download(['CL=F', 'HG=F'], start='2015-01-01', end='2025-12-31', progress=False)['Close']
macro_yf = macro_yf.rename(columns={'CL=F': 'ts_crude_oil', 'HG=F': 'ts_copper'})

macro_df = macro_fred.join(macro_yf, how='outer').ffill().dropna()
macro_df.index.name = 'session_date'

# FinBERT Data
FINBERT_PATH = DATA_DIR / "finbert_daily_df.csv"
if FINBERT_PATH.exists():
    print("Loading FinBERT data...")
    fb = pd.read_csv(FINBERT_PATH, parse_dates=["session_date"]).set_index("session_date")
    fb_col = [c for c in fb.columns if "sentiment" in c.lower()][0]
    fb_sentiment = fb[[fb_col]].rename(columns={fb_col: 'fb_sentiment'})
else:
    print("finbert_daily_df.csv not found. Generating zero-signal synthetic placeholder to complete pipeline...")
    np.random.seed(10)
    fb_sentiment = pd.DataFrame({'fb_sentiment': np.random.normal(0, 0.2, len(macro_df))}, index=macro_df.index)

market_wide_features = macro_df.join(fb_sentiment, how='inner')

def project_to_sectors(feature_wide, betas, prefix="proj"):
    rows = []
    for tkr, b_df in betas.items():
        b_df = b_df.reindex(feature_wide.index)
        w = b_df.abs().sum(axis=1).replace(0.0, np.nan)
        out = pd.DataFrame(index=feature_wide.index)
        out[f"{prefix}_score"] = (b_df * feature_wide).sum(axis=1) / w
        out["ticker"] = tkr
        rows.append(out.reset_index().rename(columns={"index": "session_date"}))
    return pd.concat(rows, ignore_index=True).dropna(subset=[f"{prefix}_score"]).reset_index(drop=True)

excess_wide = panel.pivot(index="session_date", columns="ticker", values="excess_overnight_return")

projected_dfs = []
projected_cols = []

for col in market_wide_features.columns:
    feat_wide = market_wide_features[[col]].dropna()
    betas = rolling_sector_betas(excess_wide, feat_wide, window=252)

    clean_name = col.replace('ts_', '').replace('fb_', '')
    prefix = f"proj_{clean_name}"

    proj_df = project_to_sectors(feat_wide, betas, prefix=prefix)
    projected_dfs.append(proj_df)
    projected_cols.append(f"{prefix}_score")

# Merge
for proj_df in projected_dfs:
    panel = panel.merge(proj_df, on=["session_date", "ticker"], how="left")

# Clean
panel = panel.dropna(subset=projected_cols).reset_index(drop=True)

# M0-M5 Feature Sets
FINBERT_PROJ_FEATURE = ['proj_sentiment_score']
MACRO_PROJ_FEATURES = [
    'proj_vix_score', 'proj_treasury_10y_score', 'proj_fed_funds_score',
    'proj_crude_oil_score', 'proj_copper_score', 'proj_spread_2y10y_score'
]

M0_FEATURES = MARKET_FEATURES
M1_FEATURES = MARKET_FEATURES + NARR_ON_FEATURES
M2_FEATURES = M0_FEATURES + FINBERT_PROJ_FEATURE
M3_FEATURES = M1_FEATURES + FINBERT_PROJ_FEATURE
M4_FEATURES = M3_FEATURES + MACRO_PROJ_FEATURES
M5_FEATURES = M4_FEATURES + KW_FEATURES

print("FEATURES for M0-M5")
print(f"M0_FEATURES (Market):     {len(M0_FEATURES)}")
print(f"M1_FEATURES (Narrative):  {len(M1_FEATURES)}")
print(f"M2_FEATURES (FinBERT):    {len(M2_FEATURES)}")
print(f"M3_FEATURES (Multimodal): {len(M3_FEATURES)}")
print(f"M4_FEATURES (Macro):      {len(M4_FEATURES)}")
print(f"M5_FEATURES (SectorAttr): {len(M5_FEATURES)}")
print(f"\nFinal Assembly Complete. Panel Shape: {panel.shape}")

M2-M5 MACRO & FINBERT EXPOSURE


/tmp/ipykernel_1497/435236096.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  macro_yf = yf.download(['CL=F', 'HG=F'], start='2015-01-01', end='2025-12-31', progress=False)['Close']


finbert_daily_df.csv not found. Generating zero-signal synthetic placeholder to complete pipeline...
FEATURES for M0-M5
M0_FEATURES (Market):     10
M1_FEATURES (Narrative):  16
M2_FEATURES (FinBERT):    11
M3_FEATURES (Multimodal): 17
M4_FEATURES (Macro):      23
M5_FEATURES (SectorAttr): 25

Final Assembly Complete. Panel Shape: (25831, 58)


This cell re-initializes and runs the Sector Autoformer deep learning evaluation with projected features (M0 to M4). It recovers any missing excess overnight return, defines the custom Sector Autoformer architecture, and sets up the sequence windowing and training logic. Finally, it executes the walk-forward evaluation for all defined model configurations (M0 to M4) and presents the final daily Autoformer ablation matrix.

In [ ]:
warnings.filterwarnings("ignore", category=ConstantInputWarning)

print("DEEP LEARNING SECTOR-AUTOFORMER (M0 - M5)")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LENGTH = 21
DATE, TICKER = 'session_date', 'ticker'

class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size=13):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, self.kernel_size - 1 - pad, 1)
        padded = torch.cat([front, x, end], dim=1)
        trend = self.avg(padded.permute(0, 2, 1)).permute(0, 2, 1)
        return x - trend, trend

class AutoCorrelation(nn.Module):
    def __init__(self, d_model, n_heads, factor=1, dropout=0.1):
        super().__init__()
        self.n_heads, self.d_head, self.factor = n_heads, d_model // n_heads, factor
        self.q_proj, self.k_proj = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.v_proj, self.out_proj = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _time_delay_agg(self, values, corr):
        B, H, D, L = values.shape
        top_k = max(int(self.factor * np.log(L)), 1)
        weights, delays = torch.topk(corr.mean(dim=(1, 2)), top_k, dim=-1)
        weights = torch.softmax(weights, dim=-1)
        out = torch.zeros_like(values)
        for i in range(top_k):
            d = delays[:, i]
            idx = (torch.arange(L, device=values.device).view(1, L) + d.view(B, 1)) % L
            idx = idx.view(B, 1, 1, L).expand(B, H, D, L)
            out = out + torch.gather(values, dim=-1, index=idx) * weights[:, i].view(B, 1, 1, 1)
        return out

    def forward(self, x):
        B, L, _ = x.shape
        q = self.q_proj(x).view(B, L, self.n_heads, self.d_head).permute(0, 2, 3, 1)
        k = self.k_proj(x).view(B, L, self.n_heads, self.d_head).permute(0, 2, 3, 1)
        v = self.v_proj(x).view(B, L, self.n_heads, self.d_head).permute(0, 2, 3, 1)
        q_f, k_f = torch.fft.rfft(q, dim=-1), torch.fft.rfft(k, dim=-1)
        corr = torch.fft.irfft(q_f * torch.conj(k_f), n=L, dim=-1)
        agg = self._time_delay_agg(v, corr).permute(0, 3, 1, 2).reshape(B, L, -1)
        return self.dropout(self.out_proj(agg))

class AutoformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, kernel_size=13, dropout=0.1):
        super().__init__()
        self.autocorr = AutoCorrelation(d_model, n_heads, dropout=dropout)
        self.decomp1, self.decomp2 = SeriesDecomp(kernel_size), SeriesDecomp(kernel_size)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff or 4 * d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff or 4 * d_model, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x, _ = self.decomp1(x + self.autocorr(x))
        x, _ = self.decomp2(x + self.ff(x))
        return x

class SectorAutoformer(nn.Module):
    def __init__(self, num_features, seq_length=SEQ_LENGTH, d_model=32, n_heads=4, e_layers=2, kernel_size=13, dropout=0.2):
        super().__init__()
        self.embedding = nn.Linear(num_features, d_model)
        self.pos = nn.Parameter(torch.zeros(1, seq_length, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.input_decomp = SeriesDecomp(kernel_size)
        self.layers = nn.ModuleList([AutoformerEncoderLayer(d_model, n_heads, kernel_size=kernel_size, dropout=dropout) for _ in range(e_layers)])
        self.norm, self.head = nn.LayerNorm(d_model), nn.Sequential(nn.Linear(2 * d_model, 32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        h = self.embedding(x) + self.pos
        seasonal, trend = self.input_decomp(h)
        for layer in self.layers: seasonal = layer(seasonal)
        return self.head(torch.cat([self.norm(seasonal)[:, -1, :], trend[:, -1, :]], dim=-1))

def autoformer_fit_predict(train_df, val_df, test_df, feature_cols, target_col, seed=10):
    set_seed(seed)
    scaler = StandardScaler().fit(train_df[feature_cols])
    tr, va, te = train_df.copy(), val_df.copy(), test_df.copy()
    tr[feature_cols], va[feature_cols], te[feature_cols] = scaler.transform(tr[feature_cols]), scaler.transform(va[feature_cols]), scaler.transform(te[feature_cols])

    Xtr, ytr, _ = build_sequences(tr, feature_cols, target_col)
    Xva, yva, _ = build_sequences(va, feature_cols, target_col)
    Xte, yte, idx_te = build_sequences(te, feature_cols, target_col)
    if not len(Xte): return np.full(len(test_df), float(train_df[target_col].mean()))

    model = SectorAutoformer(len(feature_cols)).to(DEVICE)
    opt, crit = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2), nn.BCEWithLogitsLoss()
    tr_dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr).unsqueeze(1)), batch_size=128, shuffle=True)

    best_auc, best_state, no_improve = -np.inf, None, 0
    for epoch in range(15):
        model.train()
        for xb, yb in tr_dl: opt.zero_grad(); crit(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        model.eval()
        with torch.no_grad(): pv = torch.sigmoid(model(torch.from_numpy(Xva).to(DEVICE))).cpu().numpy().ravel()
        auc = roc_auc_score(yva, pv) if len(np.unique(yva)) > 1 else 0.5

        if auc > best_auc + 1e-5:
            best_auc, no_improve, best_state = auc, 0, copy.deepcopy(model.state_dict())
        else:
            no_improve += 1
        if no_improve >= 5: break

    if best_state is not None: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad(): p = torch.sigmoid(model(torch.from_numpy(Xte).to(DEVICE))).cpu().numpy().ravel()

    res = idx_te.assign(p=p)
    return test_df[[TICKER, DATE]].merge(res, on=[TICKER, DATE], how="left")["p"].fillna(float(train_df[target_col].mean())).values

models_to_run = {"M0_Market_Baseline": M0_FEATURES, "M1_Narrative_Projected": M1_FEATURES, "M2_FinBERT_Projected": M2_FEATURES, "M3_Multimodal_Projected": M3_FEATURES, "M4_Macro_Projected": M4_FEATURES, "M5_Sector_Attributed": M5_FEATURES}
target_col, fwd_return_col = "target_overnight", "excess_overnight_return"

eval_df = panel.dropna(subset=[target_col, fwd_return_col] + M5_FEATURES).sort_values(['ticker', 'session_date']).reset_index(drop=True)
all_results = []
splitter = PurgedWalkForward(test_years=TEST_YEARS, label_horizon=1, embargo=1)

for name, f_set in models_to_run.items():
    print(f"  -> Training {name}")
    res = run_walk_forward(eval_df, f_set, target_col, autoformer_fit_predict, splitter, name, seeds=[42, 101, 202], fwd_return_col=fwd_return_col)

    mean_metrics = res[['ic_mean_ic', 'ic_t_stat', 'ls_gross_sharpe', 'ls_net_sharpe', 'ls_net_ann_return']].mean()
    mean_metrics.name = name
    all_results.append(mean_metrics)

print("\nDAILY AUTOFORMER MATRIX (2022-2025)")
print(pd.DataFrame(all_results).to_markdown())

This cell performs a portfolio simulation with lazy rebalancing and uses SPY as a benchmark. It extracts continuous predictions from the M4 model, assigns target portfolio weights (long top 3, short bottom 3 sectors), and applies state-change execution rules with transaction costs. Finally, it fetches the S&P 500 benchmark data, calculates cumulative returns for different strategies, and plots the performance.

In [ ]:
print("PORTFOLIO SIMULATION")

preds, dates, tickers, rets_on, rets_c2c = [], [], [], [], []
splitter = PurgedWalkForward(test_years=TEST_YEARS, label_horizon=1, embargo=1)
eval_df = panel.dropna(subset=M4_FEATURES + ["target_overnight", "excess_overnight_return", "excess_c2c_return"]).sort_values(["ticker", "session_date"])

for fold in splitter.split(eval_df):
    tr, va, te = (eval_df[eval_df["session_date"].isin(d)] for d in (fold.train_dates, fold.val_dates, fold.test_dates))
    if not len(te): continue
    p = autoformer_fit_predict(tr, va, te, M4_FEATURES, "target_overnight", seed=10)
    preds.extend(p); dates.extend(te["session_date"].values); tickers.extend(te["ticker"].values)
    rets_on.extend(te["excess_overnight_return"].values); rets_c2c.extend(te["excess_c2c_return"].values)

sim = pd.DataFrame({"session_date": pd.to_datetime(dates), "ticker": tickers, "p": preds,
                    "ret_on": rets_on, "ret_c2c": rets_c2c}).sort_values(["session_date", "ticker"])

# Long top 3 / short bottom 3 target weights / neutral everything else

sim["rank"] = sim.groupby("session_date")["p"].rank(method="first", ascending=False)
sim["w"] = np.where(sim["rank"] <= 3, 1/3, np.where(sim["rank"] >= 9, -1/3, 0.0))

W = sim.pivot(index="session_date", columns="ticker", values="w").fillna(0)
R_on = sim.pivot(index="session_date", columns="ticker", values="ret_on").fillna(0)
R_c2c = sim.pivot(index="session_date", columns="ticker", values="ret_c2c").fillna(0)

# Execution costs
cost_bps = 0.0002

# Strategy A: strict overnight (liquidate at open, re-enter at close)
turnover_strict = W.abs().sum(axis=1) * 2
net_strict = (W * R_on).sum(axis=1) - turnover_strict * cost_bps

# Strategy B: lazy state-change (hold through the day if the signal persists(C2C return))
turnover_lazy = (W - W.shift(1).fillna(0)).abs().sum(axis=1)
net_lazy = (W * R_c2c).sum(axis=1) - turnover_lazy * cost_bps

# SPY
spy = yf.download("SPY", start=sim["session_date"].min(), end=sim["session_date"].max() + pd.Timedelta(days=1), progress=False)
spy_close = spy["Close", "SPY"] if isinstance(spy.columns, pd.MultiIndex) else spy["Close"]
spy_ret = spy_close.pct_change().reindex(W.index).fillna(0)

cum_strict = (1 + net_strict).cumprod() - 1
cum_lazy = (1 + net_lazy).cumprod() - 1
cum_spy = (1 + spy_ret).cumprod() - 1

print("\nPERFORMANCE SUMMARY")
print(f"Strict Overnight Net Return: {cum_strict.iloc[-1]:.2%}")
print(f"Lazy State-Change Net Return: {cum_lazy.iloc[-1]:.2%}")
print(f"SPY Benchmark Return:        {cum_spy.iloc[-1]:.2%}")
print(f"Avg Daily Turnover (Strict): {turnover_strict.mean():.2f}")
print(f"Avg Daily Turnover (Lazy):   {turnover_lazy.mean():.2f}")

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(cum_strict.index, cum_strict, label='M4 Strict Overnight (High Turnover, Pure Alpha)', color='#00ffcc', linewidth=2)
ax.plot(cum_lazy.index, cum_lazy, label='M4 Lazy State-Change (Low Turnover, Blended Beta)', color='#ff00ff', linewidth=2, alpha=0.8)
ax.plot(cum_spy.index, cum_spy, label='SPY Benchmark', color='#ffffff', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_title('SectorAutoformer (M4) vs. S&P 500: Execution Architecture Impact', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Cumulative Net Return', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend(loc='upper left', frameon=True, fontsize=11)
plt.tight_layout()
plt.show()


This cell performs a deep learning evaluation of the M4 macro model with return decomposition ablation. It engineers strictly forward-looking targets for overnight, intraday, and close-to-close returns. It then defines a horizon matrix for these targets and executes a walk-forward evaluation using the M4 Autoformer model across these different horizons, displaying the resulting performance matrix.

In [ ]:
warnings.filterwarnings("ignore", category=ConstantInputWarning)

print("AUTOFORMER RETURN DECOMPOSITION HOLDING PERIOD (M4 MACRO)")

g = panel.groupby('ticker')
# excess_overnight_return is ALREADY T+1 (close -> next open). Do not shift.
panel['fwd_excess_overnight'] = panel['excess_overnight_return']

# Intraday and C2C are contemporaneous today, so needs to shift to T+1.
panel['fwd_excess_intraday'] = g['excess_intraday_return'].shift(-1)
panel['fwd_excess_c2c'] = g['excess_c2c_return'].shift(-1)
for col in ['overnight', 'intraday', 'c2c']:
    panel[f'target_fwd_{col}'] = (panel[f'fwd_excess_{col}'] > 0).astype(float)

horizons = {
    "1_Overnight_Gap (Close-to-Open)": ("target_fwd_overnight", "fwd_excess_overnight"),
    "2_Intraday_Session (Open-to-Close)": ("target_fwd_intraday", "fwd_excess_intraday"),
    "3_Full_Session (Close-to-Close)": ("target_fwd_c2c", "fwd_excess_c2c"),
}

eval_df_decomp = panel.dropna(subset=['target_fwd_overnight', 'target_fwd_intraday', 'target_fwd_c2c',
                                       'fwd_excess_overnight', 'fwd_excess_intraday', 'fwd_excess_c2c'] + M4_FEATURES
                              ).sort_values(['ticker', 'session_date']).reset_index(drop=True)
splitter = PurgedWalkForward(test_years=TEST_YEARS, label_horizon=1, embargo=1)

print(f"Evaluation Panel Size: {len(eval_df_decomp)} rows. Commencing Walk-Forward...")
results_decomp = []
for name, (target, fwd) in horizons.items():
    print(f"  -> Training M4 Autoformer for: {name}")
    res = run_walk_forward(eval_df_decomp, M4_FEATURES, target, autoformer_fit_predict, splitter, name,
                           seeds=[10, 11, 12], fwd_return_col=fwd)
    m = res[['ic_mean_ic', 'ic_t_stat', 'ls_gross_sharpe', 'ls_net_sharpe', 'ls_net_ann_return']].mean()
    m.name = name
    results_decomp.append(m)

print("RETURN DECOMPOSITION HOLDING PERIOD for M4 MACRO (2022-2025)")
print(pd.DataFrame(results_decomp).to_markdown())


This section introduces the evaluation of models on a monthly horizon, changing the target to a calendar month return but reverting back to a linear logistic model.

In [ ]:
print("MONTHLY LOGISTIC REGRESSION (M0 - M5)")

original_long_short_backtest = globals().get('long_short_backtest')

def monthly_long_short_backtest(pred, fwd_return, dates, n_side=3, cost_bps=0.0002):
    df = pd.DataFrame({"p": pred, "r": fwd_return, "d": dates}).dropna()
    rets, to, prev_w = [], [], None
    for d, gg in df.groupby("d"):
        if len(gg) < 2 * n_side: continue
        gg = gg.sort_values("p")
        w = pd.Series(0.0, index=gg.index)
        w.iloc[-n_side:], w.iloc[:n_side] = 1.0/n_side, -1.0/n_side
        rets.append((w * gg["r"]).sum())
        to.append((w - prev_w.reindex(w.index).fillna(0)).abs().sum() if prev_w is not None else w.abs().sum())
        prev_w = w
    r, to = np.array(rets), np.array(to)
    if len(r) < 2: return {}
    net, ann = r - to * cost_bps, np.sqrt(12)
    return {"gross_sharpe": r.mean()/r.std(ddof=1)*ann, "net_sharpe": net.mean()/net.std(ddof=1)*ann,
            "gross_ann_return": r.mean()*12, "net_ann_return": net.mean()*12}

globals()['long_short_backtest'] = monthly_long_short_backtest

panel['session_date'] = pd.to_datetime(panel['session_date'])
panel = panel.sort_values(['ticker', 'session_date'])
panel['log_c2c'] = np.log1p(panel['c2c_return'])

agg_dict = {'log_c2c': 'sum', **{f: 'last' for f in M5_FEATURES}}
monthly_panel = panel.groupby(['ticker', pd.Grouper(key='session_date', freq='ME')]).agg(agg_dict).reset_index()
monthly_panel['monthly_ret'] = np.expm1(monthly_panel['log_c2c'])
monthly_panel['excess_monthly_ret'] = monthly_panel['monthly_ret'] - monthly_panel.groupby('session_date')['monthly_ret'].transform('mean')

monthly_panel = monthly_panel.sort_values(['ticker', 'session_date'])
monthly_panel['fwd_excess_1m'] = monthly_panel.groupby('ticker')['excess_monthly_ret'].shift(-1)
monthly_panel['target_1m'] = (monthly_panel['fwd_excess_1m'] > 0).astype(float)

eval_df_m = monthly_panel.dropna(subset=['target_1m', 'fwd_excess_1m'] + M5_FEATURES).reset_index(drop=True)
print(f"Monthly Panel Size: {len(eval_df_m)} rows spanning {eval_df_m['session_date'].nunique()} months.")

models_to_run = {"M0_Market_Baseline": M0_FEATURES, "M1_Narrative_Projected": M1_FEATURES,
                 "M2_FinBERT_Projected": M2_FEATURES, "M3_Multimodal_Projected": M3_FEATURES,
                 "M4_Macro_Projected": M4_FEATURES,
                 "M5_Sector_Attributed": M5_FEATURES}
splitter = PurgedWalkForward(test_years=TEST_YEARS, label_horizon=1, embargo=1)

print("\nMonthly Walk-Forward (Logistic)")
monthly_results = []
for name, f_set in models_to_run.items():
    print(f"  -> Training {name} ({len(f_set)} features)")
    res = run_walk_forward(eval_df_m, f_set, 'target_1m', example_logistic_fit_predict, splitter, name,
                           seeds=[10, 11, 12, 13, 14], fwd_return_col='fwd_excess_1m')
    m = res[['ic_mean_ic', 'ic_t_stat', 'ls_gross_sharpe', 'ls_net_sharpe', 'ls_net_ann_return']].mean()
    m.name = name
    monthly_results.append(m)

if original_long_short_backtest is not None: globals()['long_short_backtest'] = original_long_short_backtest

print("MONTHLY LOGISTIC PERFORMANCE MATRIX (2022-2025)")
print(pd.DataFrame(monthly_results).to_markdown())

**Monthly Autoformer**

This cell evaluates models on a monthly horizon using a calendar month target. It isolates the last trading day of each month, engineers monthly returns and a forward target. It then defines an ablation loop for M0, M1, M2, M3, M4 and M5 using a monthly-adapted Sector Autoformer. It runs walk-forward evaluations for each model and displays the final monthly ablation matrix.

In [ ]:
print("MONTHLY AUTOFORMER MATRIX (M0 - M5)")

original_long_short_backtest = globals().get('long_short_backtest')

def monthly_long_short_backtest(pred, fwd_return, dates, n_side=3, cost_bps=0.0002):
    df = pd.DataFrame({"p": pred, "r": fwd_return, "d": dates}).dropna()
    rets, to, prev_w = [], [], None
    for d, gg in df.groupby("d"):
        if len(gg) < 2 * n_side: continue
        gg = gg.sort_values("p")
        w = pd.Series(0.0, index=gg.index)
        w.iloc[-n_side:], w.iloc[:n_side] = 1.0/n_side, -1.0/n_side
        rets.append((w * gg["r"]).sum())
        to.append((w - prev_w.reindex(w.index).fillna(0)).abs().sum() if prev_w is not None else w.abs().sum())
        prev_w = w
    r, to = np.array(rets), np.array(to)
    if len(r) < 2: return {}
    net, ann = r - to * cost_bps, np.sqrt(12)
    return {"gross_sharpe": r.mean()/r.std(ddof=1)*ann, "net_sharpe": net.mean()/net.std(ddof=1)*ann,
            "gross_ann_return": r.mean()*12, "net_ann_return": net.mean()*12}

globals()['long_short_backtest'] = monthly_long_short_backtest

MONTHLY_SEQ_LENGTH = 12

def build_monthly_sequences(df, feature_cols, target_col, seq_length=MONTHLY_SEQ_LENGTH):
    seqs, labels, keys = [], [], []
    df = df.sort_values(['ticker', 'session_date'])
    for tkr, gg in df.groupby('ticker', sort=False):
        X, y, d = gg[list(feature_cols)].values, gg[target_col].values, gg['session_date'].values
        if len(gg) < seq_length: continue
        for end in range(seq_length - 1, len(gg)):
            win = X[end-seq_length+1:end+1]
            if np.isnan(y[end]) or not np.isfinite(win).all(): continue
            seqs.append(win); labels.append(y[end]); keys.append((tkr, d[end]))
    return np.array(seqs, np.float32), np.array(labels, np.float32), pd.DataFrame(keys, columns=['ticker', 'session_date'])

def monthly_autoformer_fit_predict(train_df, val_df, test_df, feature_cols, target_col, seed=10):
    set_seed(seed)
    scaler = StandardScaler().fit(train_df[feature_cols])
    tr, va = train_df.copy(), val_df.copy()
    tr[feature_cols], va[feature_cols] = scaler.transform(tr[feature_cols]), scaler.transform(va[feature_cols])

    context = va.groupby('ticker').tail(MONTHLY_SEQ_LENGTH - 1)
    te = pd.concat([context, test_df.copy()]).sort_values(['ticker', 'session_date'])
    te[feature_cols] = scaler.transform(te[feature_cols])

    Xtr, ytr, _ = build_monthly_sequences(tr, feature_cols, target_col)
    Xva, yva, _ = build_monthly_sequences(va, feature_cols, target_col)
    Xte, yte, idx_te = build_monthly_sequences(te, feature_cols, target_col)
    if not len(Xte): return np.full(len(test_df), float(train_df[target_col].mean()))

    model = SectorAutoformer(len(feature_cols), seq_length=MONTHLY_SEQ_LENGTH).to(DEVICE)
    opt, crit = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2), nn.BCEWithLogitsLoss()
    tr_dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr).unsqueeze(1)), batch_size=128, shuffle=True)

    best_auc, best_state, no_improve = -np.inf, None, 0
    for epoch in range(15):
        model.train()
        for xb, yb in tr_dl: opt.zero_grad(); crit(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        model.eval()
        with torch.no_grad(): pv = torch.sigmoid(model(torch.from_numpy(Xva).to(DEVICE))).cpu().numpy().ravel()
        auc = roc_auc_score(yva, pv) if len(np.unique(yva)) > 1 else 0.5
        if auc > best_auc + 1e-5: best_auc, no_improve, best_state = auc, 0, copy.deepcopy(model.state_dict())
        else: no_improve += 1
        if no_improve >= 5: break
    if best_state is not None: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad(): p = torch.sigmoid(model(torch.from_numpy(Xte).to(DEVICE))).cpu().numpy().ravel()

    res = idx_te.assign(p=p)
    return test_df[['ticker', 'session_date']].merge(res, on=['ticker', 'session_date'], how='left')['p'].fillna(float(train_df[target_col].mean())).values

models_to_run = {"M0_Market_Baseline": M0_FEATURES, "M1_Narrative_Projected": M1_FEATURES,
                 "M2_FinBERT_Projected": M2_FEATURES, "M3_Multimodal_Projected": M3_FEATURES,
                 "M4_Macro_Projected": M4_FEATURES,
                 "M5_Sector_Attributed": M5_FEATURES}
splitter = PurgedWalkForward(test_years=TEST_YEARS, label_horizon=1, embargo=1)

print("\nMonthly Walk-Forward (Sector Autoformer)")
monthly_results_dl = []
for name, f_set in models_to_run.items():
    print(f"  -> Training {name} ({len(f_set)} features)")
    res = run_walk_forward(eval_df_m, f_set, 'target_1m', monthly_autoformer_fit_predict, splitter, name,
                           seeds=[10, 11, 12], fwd_return_col='fwd_excess_1m')
    m = res[['ic_mean_ic', 'ic_t_stat', 'ls_gross_sharpe', 'ls_net_sharpe', 'ls_net_ann_return']].mean()
    m.name = name
    monthly_results_dl.append(m)

if original_long_short_backtest is not None: globals()['long_short_backtest'] = original_long_short_backtest

print("MONTHLY AUTOFORMER PERFORMANCE MATRIX (2022-2025)")
print(pd.DataFrame(monthly_results_dl).to_markdown())
